<div class='alert alert-warning'>

# JupyterLite warning

Running the scikit-learn examples in JupyterLite is experimental and you may encounter some unexpected behavior.

The main difference is that imports will take a lot longer than usual, for example the first `import sklearn` can take roughly 10-20s.

If you notice problems, feel free to open an [issue](https://github.com/nilearn/nilearn/issues/new/choose) about it.
</div>

In [ ]:
# JupyterLite-specific code
%pip install pyodide-http
import pyodide_http
pyodide_http.patch_all()
import matplotlib
import pandas


# Visualizing global patterns with a carpet plot

A common quality control step for functional MRI data is to visualize the data
over time in a carpet plot (also known as a Power plot or a grayplot).

The :func:`~nilearn.plotting.plot_carpet()` function generates a carpet plot
from a 4D functional image.


## Fetching data from ADHD dataset



In [ ]:
from nilearn.datasets import fetch_adhd
from nilearn.plotting import plot_carpet

adhd_dataset = fetch_adhd(n_subjects=1)

# Print basic information on the dataset
print(
    f"First subject functional nifti image (4D) is at: {adhd_dataset.func[0]}"
)

## Deriving a mask



In [ ]:
from nilearn import masking

# Build an EPI-based mask because we have no anatomical data
mask_img = masking.compute_epi_mask(adhd_dataset.func[0])

## Visualizing global patterns over time



In [ ]:
import matplotlib.pyplot as plt

display = plot_carpet(
    adhd_dataset.func[0],
    mask_img,
    t_r=adhd_dataset.t_r,
    title="global patterns over time",
)

display.show()

## Deriving a label-based mask
Create a gray matter/white matter/cerebrospinal fluid mask from
ICBM152 tissue probability maps.



In [ ]:
import numpy as np

from nilearn import image
from nilearn.datasets import fetch_icbm152_2009

atlas = fetch_icbm152_2009()
atlas_img = image.concat_imgs((atlas["gm"], atlas["wm"], atlas["csf"]))
map_labels = {"Gray Matter": 1, "White Matter": 2, "Cerebrospinal Fluid": 3}

atlas_data = atlas_img.get_fdata()
discrete_version = np.argmax(atlas_data, axis=3) + 1
discrete_version[np.max(atlas_data, axis=3) == 0] = 0
discrete_atlas_img = image.new_img_like(
    atlas_img, discrete_version.astype(np.float32)
)

## Visualizing global patterns, separated by tissue type



In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

display = plot_carpet(
    adhd_dataset.func[0],
    discrete_atlas_img,
    t_r=adhd_dataset.t_r,
    mask_labels=map_labels,
    axes=ax,
    title="global patterns over time separated by tissue type",
)

plt.show()